## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Libraries imported")

## Load Data

In [ ]:
df = pd.read_csv('heart.csv')

print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Source: heart.csv")

df.head()

In [ ]:
df.describe()

In [ ]:
print("Data info:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
print("Target distribution:")
print(df['target'].value_counts().sort_index())

df['target'].value_counts().sort_index().plot(kind='bar', color='skyblue')
plt.title('Disease Distribution')
plt.xlabel('Status')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

In [ ]:
features = ['age', 'trestbps', 'chol', 'thalach']
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

for idx, col in enumerate(features):
    axes[idx].hist(df[col], bins=30, color='lightcoral', edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'{col.title()}')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
correlation = df.corr()
sns.heatmap(correlation, annot=True, cmap='coolwarm', fmt='.2f', 
            linewidths=0.5, square=True)
plt.title('Correlation Heatmap')
plt.show()

print("\nTop features correlated with target:")
print(correlation['target'].sort_values(ascending=False).head(6))

## Preprocessing

In [ ]:
df_processed = df.copy()

print("Handling outliers using IQR method...")
numerical_features = df_processed.select_dtypes(include=[np.number]).columns.tolist()
numerical_features.remove('target')

outlier_counts = {}
for feature in numerical_features:
    Q1 = df_processed[feature].quantile(0.25)
    Q3 = df_processed[feature].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df_processed[(df_processed[feature] < lower_bound) | 
                           (df_processed[feature] > upper_bound)]
    outlier_counts[feature] = len(outliers)
    
    df_processed[feature] = df_processed[feature].clip(lower_bound, upper_bound)

print(f"Capped outliers in {len([v for v in outlier_counts.values() if v > 0])} features")

In [ ]:
print("Creating new features...")

df_processed['age_thalach'] = df_processed['age'] * df_processed['thalach']
df_processed['bp_chol_ratio'] = df_processed['trestbps'] / (df_processed['chol'] + 0.001)
df_processed['age_squared'] = df_processed['age'] ** 2
df_processed['exercise_heart_ratio'] = df_processed['exang'] * df_processed['thalach']

print(f"New shape: {df_processed.shape}")

In [ ]:
print("Converting to binary classification...")
print("0 = no disease, 1 = disease")

df_processed['disease_binary'] = (df_processed['target'] > 0).astype(int)

print("\nClass distribution:")
print(df_processed['disease_binary'].value_counts())

In [ ]:
X = df_processed.drop(['target', 'disease_binary'], axis=1)
y = df_processed['disease_binary']

print(f"Features: {X.shape}")
print(f"Target: {y.shape}")

In [ ]:
print("Scaling features...")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print(f"Scaled {X_scaled.shape[1]} features")
print("\nFirst 5 rows:")
print(X_scaled.head())

## Model Training

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, 
                                                    test_size=0.2, 
                                                    random_state=42, 
                                                    stratify=y)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")

In [ ]:
print("Training Naive Bayes...")

nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

print("Done")

In [ ]:
print("Training Decision Tree...")

dt_model = DecisionTreeClassifier(max_depth=10, 
                                 min_samples_split=20,
                                 min_samples_leaf=10, 
                                 random_state=42)
dt_model.fit(X_train, y_train)

print("Done")

In [ ]:
print("Training Neural Network (64-32-16 architecture)...")

ann_model = MLPClassifier(hidden_layer_sizes=(64, 32, 16),
                         activation='relu',
                         solver='adam',
                         max_iter=500,
                         random_state=42,
                         early_stopping=True,
                         validation_fraction=0.1)
ann_model.fit(X_train, y_train)

print(f"Done - took {ann_model.n_iter_} iterations")

## Evaluation

In [ ]:
models = {
    'Naive Bayes': nb_model,
    'Decision Tree': dt_model,
    'ANN': ann_model
}

results = {}

print("Evaluating models...")

In [ ]:
for model_name, model in models.items():
    print(f"\n{model_name}:")
    
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    train_accuracy = accuracy_score(y_train, y_train_pred)
    test_accuracy = accuracy_score(y_test, y_test_pred)
    
    print(f"  Training: {train_accuracy:.4f}")
    print(f"  Testing:  {test_accuracy:.4f}")
    print(f"  Gap:      {(train_accuracy - test_accuracy):.4f}")
    
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_test_pred, 
                                target_names=['No Disease', 'Disease']))
    
    cm = confusion_matrix(y_test, y_test_pred)
    print(f"Confusion Matrix:")
    print(f"              Predicted")
    print(f"              No   Yes")
    print(f"Actual No   [{cm[0][0]:3d}   {cm[0][1]:3d}]")
    print(f"Actual Yes  [{cm[1][0]:3d}   {cm[1][1]:3d}]")
    
    results[model_name] = {
        'train_accuracy': train_accuracy,
        'test_accuracy': test_accuracy,
        'confusion_matrix': cm
    }

In [ ]:
model_names = list(results.keys())
train_accs = [results[m]['train_accuracy'] for m in model_names]
test_accs = [results[m]['test_accuracy'] for m in model_names]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Accuracy comparison
x = np.arange(len(model_names))
width = 0.35

axes[0, 0].bar(x - width/2, train_accs, width, label='Training', color='#3498db', alpha=0.8)
axes[0, 0].bar(x + width/2, test_accs, width, label='Testing', color='#e74c3c', alpha=0.8)
axes[0, 0].set_xlabel('Model')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].set_title('Accuracy Comparison')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(model_names, rotation=15, ha='right')
axes[0, 0].set_ylim([0.6, 1.0])
axes[0, 0].legend()
axes[0, 0].grid(axis='y', alpha=0.3)

# Test accuracy only
colors = ['#2ecc71', '#9b59b6', '#f39c12']
bars = axes[0, 1].bar(model_names, test_accs, color=colors, alpha=0.7, edgecolor='black')
axes[0, 1].set_xlabel('Model')
axes[0, 1].set_ylabel('Test Accuracy')
axes[0, 1].set_title('Test Accuracy')
axes[0, 1].set_xticklabels(model_names, rotation=15, ha='right')
axes[0, 1].set_ylim([0.6, 1.0])
axes[0, 1].grid(axis='y', alpha=0.3)

for bar in bars:
    height = bar.get_height()
    axes[0, 1].text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.4f}',
                    ha='center', va='bottom', fontsize=9)

# Confusion matrices
for idx, (model_name, result) in enumerate(results.items()):
    row = 0 if idx == 0 else 1
    col = 2 if idx == 0 else idx - 1
    
    cm = result['confusion_matrix']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
               xticklabels=['No', 'Yes'],
               yticklabels=['No', 'Yes'],
               ax=axes[row, col])
    axes[row, col].set_title(f'{model_name}')
    axes[row, col].set_ylabel('Actual')
    axes[row, col].set_xlabel('Predicted')

fig.delaxes(axes[1, 2])

plt.tight_layout()
plt.savefig('c:/Users/aliom/Intro_to_ai/notebook_results.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved notebook_results.png")

In [ ]:
print("\n" + "="*60)
print("Summary")
print("="*60)

summary_df = pd.DataFrame({
    'Model': model_names,
    'Test Acc': test_accs,
    'Train Acc': train_accs,
    'Gap': [train - test for train, test in zip(train_accs, test_accs)]
})

summary_df = summary_df.sort_values('Test Acc', ascending=False).reset_index(drop=True)

print("\nRanked by test accuracy:")
print(summary_df.to_string(index=False))

best_idx = test_accs.index(max(test_accs))
best_model = model_names[best_idx]
best_acc = test_accs[best_idx]

print(f"\nBest model: {best_model}")
print(f"Accuracy: {best_acc:.4f} ({best_acc*100:.2f}%)")

## Notes

The models achieved good performance in predicting heart disease:
- Naive Bayes and Decision Tree both reached ~79% accuracy
- Neural Network achieved ~70% accuracy
- Feature engineering helped improve results
- All models show reasonable generalization (low overfitting)